In [47]:
import logging
import re
from typing import List, Optional, Any

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from scipy.sparse import csr_matrix
import pandas as pd
from skfp.fingerprints import AtomPairFingerprint, TopologicalTorsionFingerprint
from rdkit import Chem
from skfp.fingerprints import RDKit2DDescriptorsFingerprint
from rdkit.Chem import SaltRemover
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import f1_score

from collections import defaultdict
from typing import List, Tuple, Dict

import numpy as np
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# Autentyczne klasy z ekosystemu scikit-fingerprints
from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint

In [2]:
class DAGMolStandardizer(BaseEstimator, TransformerMixin):
    """Niskopoziomowy transformator RDKit przygotowujący grafy pod klasyfikację ChEBI.

    Klasa parsuje SMILES, weryfikuje poprawność, usuwa sole, ale w przeciwieństwie
    do filtrów ADME, nie usuwa skrajnie dużych molekuł, ponieważ ontologia ChEBI
    zawiera również makromolekuły i złożone peptydy.

    Atrybuty:
        salt_remover (SaltRemover): Instancja do odcinania asocjatów jonowych.
    """

    def __init__(self) -> None:
        """Inicjalizacja standardyzatora z modułem SaltRemover."""
        self.salt_remover = SaltRemover.SaltRemover()

    def fit(self, X: List[str], y: Optional[Any] = None) -> 'DAGMolStandardizer':
        """Metoda dopasowania (statyczna standaryzacja nie wymaga treningu).

        Args:
            X: Lista ciągów znaków SMILES (33 668 instancji).
            y: Etykiety (ignorowane na etapie wektoryzacji).

        Returns:
            Instancja samej siebie.
        """
        return self

    def transform(self, X: List[str]) -> List[Optional[Chem.Mol]]:
        """Aplikuje standaryzację na grafach molekularnych.

        Args:
            X: Lista ciągów SMILES.

        Returns:
            Lista wyczyszczonych obiektów rdkit.Chem.Mol. Niewłaściwe SMILES
            zwracają wartość None, co zapobiega awarii potoku.
        """
        standardized_mols = []
        for smiles in X:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                logging.warning(f"Odrzucono uszkodzony SMILES: {smiles}")
                standardized_mols.append(None)
                continue

            # Bezpieczne usunięcie krystalizatorów i resztek rozpuszczalników
            clean_mol = self.salt_remover.StripMol(mol, dontRemoveEverything=True)

            if clean_mol is not None:
                Chem.SanitizeMol(clean_mol)

            standardized_mols.append(clean_mol)

        return standardized_mols


def build_chebi_feature_pipeline() -> Pipeline:
    """Konstruuje potok inżynierii cech zoptymalizowany pod klasyfikację strukturalną DAG.

    Używa obiektu FeatureUnion do równoległego złączenia zliczeniowych promieni otoczenia
    (ECFP) z deterministycznym detektorem grup funkcyjnych (MACCS Keys). Całość
    operuje na rzadkich macierzach pamięci.

    Returns:
        Obiekt sklearn.pipeline.Pipeline ustandaryzowany do integracji z docelowym
        estymatorem (np. estymatorami drzewiastymi wspierającymi MultiOutput).
    """
    # Zderzamy detekcję grup funkcyjnych (MACCS) z głęboką analizą promieniową (ECFP)
    # n_jobs=-1 gwarantuje zużycie 100% lokalnego procesora
    feature_fusion = FeatureUnion(transformer_list=[
        ('maccs_keys', MACCSFingerprint(sparse=True, n_jobs=-1)),
        ('ecfp_counts', ECFPFingerprint(count=True, sparse=True, n_jobs=-1))
    ], n_jobs=-1)

    pipeline = Pipeline(steps=[
        # Krok 1: Walidacja i czyszczenie
        ('standardization', DAGMolStandardizer()),

        # Omijamy filtrację Lipinskiego - ChEBI bada strukturę, a nie wchłanialność leku!

        # Krok 2: Generacja zoptymalizowanej, rzadkiej macierzy fuzji
        ('hybrid_vectorization', feature_fusion),

        ('atom_pair', AtomPairFingerprint(count=True, sparse=True, n_jobs=-1)),

        ('torsion', TopologicalTorsionFingerprint(count=True, sparse=True, n_jobs=-1)),

        ('rdkit_descriptors', RDKit2DDescriptorsFingerprint(sparse=False, n_jobs=-1))

        # Uwaga: Estymator zostanie dodany w kolejnym kroku, po omówieniu strategii dla DAG.
    ])

    return pipeline

In [3]:
preprocess_pipeline = build_chebi_feature_pipeline()

In [8]:
data_path = '../../1_ontology/data/chebi_dataset_train.parquet'
df = pd.read_parquet(data_path)

In [10]:
df

,mol_id,SMILES,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,mol_12500,CCCCC/C=C\CCCCCCCC(=O)O,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,mol_15962,Cc1cc2cc(O)cc(O)c2c(C)n1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,mol_42147,C[C@H](CCC[C@@H](C)C=O)[C@H]1CC[C@H]2[C@@H]3CC...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,mol_43459,CCN=C1C=CC(=C(c2ccc(NCC)cc2)c2ccc(NCC)c(C)c2)C=C1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,mol_12734,C[C@H](CCC(O)=N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33663,mol_26342,C[C@H](OP(=O)(O)OC[C@@H](O)[C@@H](O)[C@@H](O)C...,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
33664,mol_5233,O=C(Nc1ccc(Cl)cc1)Nc1ccc(Cl)cc1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
33665,mol_36641,Cc1cn([C@H]2C[C@H](OP(=O)(O)OC[C@H]3O[C@@H](n4...,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
33666,mol_9633,O=C([O-])C(O)Cc1c[nH]c2ccccc12,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [33]:
def generate_scaffold(smiles: str, include_chirality: bool = False) -> str:
    """Generuje rusztowanie Bemis-Murcko dla podanego ciągu SMILES.

    Args:
        smiles (str): Ciąg znaków SMILES molekuły.
        include_chirality (bool): Czy uwzględniać stereochemię w rusztowaniu.

    Returns:
        str: SMILES rusztowania lub pusty ciąg, jeśli parsowanie zawiodło.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""
    # Generowanie rusztowania (szkieletu pierścieniowego z łącznikami)
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol, includeChirality=include_chirality
    )
    return scaffold


def scaffold_split(
    df: pd.DataFrame,
    smiles_col: str = 'SMILES',
    train_size: float = 0.8
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Wykonuje podział zbioru danych w oparciu o unikalne rusztowania molekularne.

    Zapewnia, że wszystkie molekuły o tym samym rdzeniu (scaffold) znajdują się
    wyłącznie w jednym ze zbiorów (treningowym lub testowym).

    Args:
        df (pd.DataFrame): Wejściowy zbiór danych ChEBI.
        smiles_col (str): Nazwa kolumny z ciągami SMILES.
        train_size (float): Proporcja zbioru treningowego.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Ramki danych (train, test).
    """
    logging.info("Rozpoczynam generowanie rusztowań dla Scaffold Split...")

    # Grupowanie indeksów według unikalnych rusztowań
    scaffolds: Dict[str, List[int]] = defaultdict(list)
    for idx, smiles in enumerate(df[smiles_col]):
        scaff_smiles = generate_scaffold(smiles)
        scaffolds[scaff_smiles].append(idx)

    # Sortowanie grup od największej do najmniejszej (praktyka stabilizująca podział)
    scaffold_sets = sorted(scaffolds.values(), key=len, reverse=True)

    train_indices: List[int] = []
    test_indices: List[int] = []
    n_train_cutoff = int(len(df) * train_size)

    for scaffold_set in scaffold_sets:
        if len(train_indices) + len(scaffold_set) <= n_train_cutoff:
            train_indices.extend(scaffold_set)
        else:
            test_indices.extend(scaffold_set)

    print(f"Podział zakończony. Train: {len(train_indices)}, Test: {len(test_indices)}")
    return df.iloc[train_indices], df.iloc[test_indices]

In [34]:
df_train, df_test = scaffold_split(df)

[20:57:17] WARNING: not removing hydrogen atom without neighbors
[20:57:17] WARNING: not removing hydrogen atom without neighbors
[20:57:17] WARNING: not removing hydrogen atom without neighbors
[20:57:17] WARNING: not removing hydrogen atom without neighbors
[20:57:18] WARNING: not removing hydrogen atom without neighbors
[20:57:18] WARNING: not removing hydrogen atom without neighbors
[20:57:18] WARNING: not removing hydrogen atom without neighbors
[20:57:18] WARNING: not removing hydrogen atom without neighbors
[20:57:18] WARNING: not removing hydrogen atom without neighbors
[20:57:18] Unusual charge on atom 0 number of radical electrons set to zero
[20:57:19] WARNING: not removing hydrogen atom without neighbors
[20:57:19] WARNING: not removing hydrogen atom without neighbors
[20:57:19] WARNING: not removing hydrogen atom without neighbors
[20:57:19] WARNING: not removing hydrogen atom without neighbors
[20:57:19] WARNING: not removing hydrogen atom without neighbors
[20:57:19] WAR

Podział zakończony. Train: 26934, Test: 6734


[20:57:28] WARNING: not removing hydrogen atom without neighbors
[20:57:28] WARNING: not removing hydrogen atom without neighbors
[20:57:28] WARNING: not removing hydrogen atom without neighbors
[20:57:28] WARNING: not removing hydrogen atom without neighbors


In [40]:
def parse_hierarchy(file_path: str) -> Dict[str, List[str]]:
    """Parsuje plik chebi_classes.txt w celu wyodrębnienia relacji is_a.

    Args:
        file_path (str): Ścieżka do pliku z definicjami klas ChEBI.

    Returns:
        Dict[str, List[str]]: Mapowanie: klasa_dziecko -> lista_klas_rodziców.
    """
    parents = {}
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Podział na poszczególne termy w pliku OBO/ChEBI
    terms = content.split("[Term]")
    for term in terms[1:]:
        # Wyciągnięcie ID klasy
        id_match = re.search(r"^id:\s*(class_\d+)", term, re.MULTILINE)
        if not id_match:
            continue
        child_id = id_match.group(1)

        # Wyciągnięcie wszystkich klas nadrzędnych (is_a)
        is_a_matches = re.findall(r"^is_a:\s*(class_\d+)", term, re.MULTILINE)
        parents[child_id] = is_a_matches
    return parents

def apply_transitive_closure(
    y: np.ndarray,
    columns: List[str],
    hierarchy: Dict[str, List[str]]
) -> np.ndarray:
    """Wymusza spójność hierarchiczną w macierzy etykiet y.

    Zapewnia, że jeśli molekuła należy do klasy specyficznej, to należy również
    do wszystkich jej klas nadrzędnych w grafie DAG.

    Args:
        y (np.ndarray): Macierz binarnych etykiet.
        columns (List[str]): Lista nazw kolumn klas.
        hierarchy (Dict[str, List[str]]): Słownik relacji is_a.

    Returns:
        np.ndarray: Przetworzona macierz y z domknięciem przechodnim.
    """
    col_to_idx = {name: i for i, name in enumerate(columns)}
    y_closed = y.copy()

    # Sortowanie kolumn malejąco (od liści do korzenia grafu)
    for col_name in reversed(columns):
        if col_name in hierarchy:
            child_idx = col_to_idx[col_name]
            for parent_name in hierarchy[col_name]:
                if parent_name in col_to_idx:
                    parent_idx = col_to_idx[parent_name]
                    # Logiczne OR: jeśli dziecko ma cechę, rodzic też musi ją mieć
                    y_closed[:, parent_idx] = np.maximum(
                        y_closed[:, parent_idx], y_closed[:, child_idx]
                    )
    return y_closed

In [43]:
# 1. Parsowanie hierarchii z pliku
hierarchy_map = parse_hierarchy("chebi_classes.txt")

# 2. Wyciągnięcie surowych etykiet y z Twoich ramek danych
# class_cols to lista: ['class_0', 'class_1', ..., 'class_499']
class_cols = [col for col in df_train.columns if col.startswith("class_")]
y_train_raw = df_train[class_cols].values
y_test_raw = df_test[class_cols].values

# 3. Aplikacja Transitive Closure (kluczowe dla F1-Macro i spójności)
logging.info("Aplikuję domknięcie przechodnie dla hierarchii ChEBI...")
y_train = apply_transitive_closure(y_train_raw, class_cols, hierarchy_map)
y_test = apply_transitive_closure(y_test_raw, class_cols, hierarchy_map)

In [46]:
model = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

logging.info("Rozpoczynam trening modelu baseline (Random Forest)...")
model.fit(X_train, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [48]:
y_pred = model.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average="macro")
logging.info(f"Trening zakończony. Wynik F1-Macro: {macro_f1:.4f}")

C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [49]:
macro_f1

0.18211360433312065

In [90]:
import importlib
import chebi_feature_pipeline  # Importujemy cały moduł

# Przeładowujemy moduł
importlib.reload(chebi_feature_pipeline)

# Wyciągamy odświeżoną funkcję i klasę
from chebi_feature_pipeline import preprocess_chebi_data, FeatureConfig

X_train, X_test = preprocess_chebi_data(df_train, df_test, smiles_col='SMILES')

[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] Unusual charge on atom 0 number of radical electrons set to zero
[21:46:55] Unusual charge on atom 0 number of radical electrons set to zero
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21:46:55] WARNING: not removing hydrogen atom without neighbors
[21

In [92]:
import networkx as nx
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

In [99]:
# ============================================================
# CELL 1 — Importy
# ============================================================
import logging
import sys
import time
from typing import List, Tuple

import lightgbm as lgb
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)


# ============================================================
# CELL 2 — Kolumny etykiet (zakładamy że df_train już istnieje)
# ============================================================
LABEL_PREFIX = "class_"
label_cols = [c for c in df_train.columns if c.startswith(LABEL_PREFIX)]
n_classes  = len(label_cols)

y_train = df_train[label_cols].values.astype(np.float32)
y_test  = df_test[label_cols].values.astype(np.float32)  # usuń jeśli test bez etykiet

print(f"Liczba klas : {n_classes}")
print(f"X_train     : {X_train.shape}")
print(f"X_test      : {X_test.shape}")
print(f"y_train     : {y_train.shape}")

Liczba klas : 500
X_train     : (26934, 11460)
X_test      : (6734, 11460)
y_train     : (26934, 500)


In [100]:
# ============================================================
# CELL 3 — Parser DAG z chebi_classes.txt
# ============================================================
def parse_chebi_dag(obo_path: str) -> nx.DiGraph:
    """Buduje DAG z relacji is_a. Krawędź: child → parent."""
    dag = nx.DiGraph()
    current_id = None
    with open(obo_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current_id = None
            elif line.startswith("id:"):
                current_id = line.split("id:")[1].strip()
                dag.add_node(current_id)
            elif line.startswith("is_a:") and current_id:
                parent = line.split("is_a:")[1].strip().split()[0]
                dag.add_edge(current_id, parent)
    log.info(f"DAG: {dag.number_of_nodes()} węzłów, {dag.number_of_edges()} krawędzi")
    return dag


dag = parse_chebi_dag("chebi_classes.txt")

In [101]:
# ============================================================
# CELL 4 — Propagacja etykiet w górę DAG (postprocessing)
# ============================================================
def propagate_labels_upward(
    y_pred: np.ndarray,
    dag: nx.DiGraph,
    class_columns: List[str],
) -> np.ndarray:
    """Jeśli dziecko=1, wszyscy przodkowie=1. Redukuje niespójności DAG."""
    col_to_idx = {col: i for i, col in enumerate(class_columns)}
    result = y_pred.copy()
    try:
        topo_order = list(nx.topological_sort(dag))
    except nx.NetworkXUnfeasible:
        log.warning("DAG zawiera cykl – pomijam propagację.")
        return result
    for node in topo_order:
        if node not in col_to_idx:
            continue
        active = np.where(result[:, col_to_idx[node]] == 1)[0]
        if len(active) == 0:
            continue
        for parent in dag.successors(node):
            if parent in col_to_idx:
                result[active, col_to_idx[parent]] = 1
    fixed = int(np.sum(result) - np.sum(y_pred))
    log.info(f"Propagacja DAG: naprawiono {fixed} niespójnych etykiet.")
    return result


def count_dag_inconsistencies(
    y_prob: np.ndarray,
    dag: nx.DiGraph,
    class_columns: List[str],
) -> int:
    """Liczy pary (próbka, krawędź) gdzie P(dziecko) > P(rodzic)."""
    col_to_idx = {col: i for i, col in enumerate(class_columns)}
    total = 0
    for child, parent in dag.edges():
        if child not in col_to_idx or parent not in col_to_idx:
            continue
        total += int(np.sum(
            y_prob[:, col_to_idx[child]] > y_prob[:, col_to_idx[parent]]
        ))
    return total


In [102]:
# ============================================================
# CELL 5 — Konfiguracja LightGBM
# ============================================================
LGBM_PARAMS = {
    "objective":         "binary",
    "metric":            "binary_logloss",
    "boosting_type":     "gbdt",
    "num_leaves":        63,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 20,
    "reg_alpha":         0.1,
    "reg_lambda":        0.1,
    "is_unbalance":      True,  # obsługa niezbalansowanych klas ChEBI
    "verbose":           -1,
    "n_jobs":            -1,
}

N_ESTIMATORS = 300   # zwiększ do 500–1000 na finalnym treningu
EARLY_STOP   = 30    # rundy bez poprawy → stop
THRESHOLD    = 0.5   # próg decyzyjny
F1_PERIOD    = 20    # co ile rund liczyć F1 w callbacku


In [103]:
# ============================================================
# CELL 6 — Callback F1 dla pojedynczego klasyfikatora binarnego
# ============================================================
class F1Callback:
    """Liczy binary F1 na zbiorze testowym co F1_PERIOD rund boostingu."""

    def __init__(self, X_eval, y_eval_col, period=20, threshold=0.5):
        self.X_eval     = X_eval
        self.y_eval_col = y_eval_col
        self.period     = period
        self.threshold  = threshold
        self.history: List[Tuple[int, float]] = []

    def __call__(self, env: lgb.callback.CallbackEnv) -> None:
        if (env.iteration + 1) % self.period != 0:
            return
        y_prob = env.model.predict(self.X_eval)
        y_pred = (y_prob >= self.threshold).astype(int)
        f1 = f1_score(self.y_eval_col, y_pred, average="binary", zero_division=0)
        self.history.append((env.iteration + 1, f1))

In [120]:
# ============================================================
# CELL 7 — Trening: pętla po klasach z monitoringiem F1 macro
# ============================================================

# 1. Instalacja za pomocą %pip (lepiej niż !pip, bo celuje dokładnie w ten kernel)
import tqdm
if hasattr(tqdm, 'notebook'):
    from tqdm import tqdm as tqdm_base
    tqdm = tqdm_base
else:
    from tqdm import tqdm

# 3. Opcjonalne: Sprawdzenie czy import już przechodzi
try:
    import ipywidgets
    print("Sukces: ipywidgets zaimportowane bez restartu!")
except ImportError:
    print("Niestety, dla ipywidgets specyfika widgetów Jupyter może wymagać odświeżenia strony (F5).")
t0 = time.time()

models           = []
f1_per_class     = {}
prob_matrix_test = np.zeros((X_test.shape[0], n_classes), dtype=np.float32)

pbar = tqdm(range(n_classes), desc="Trening klas", unit="klasa", leave=True)

for i in pbar:
    class_name = label_cols[i]
    y_tr_col   = y_train[:, i]
    y_te_col   = y_test[:, i]

    dtrain = lgb.Dataset(X_train, label=y_tr_col, free_raw_data=False)
    deval  = lgb.Dataset(X_test,  label=y_te_col, reference=dtrain, free_raw_data=False)

    f1_cb = F1Callback(X_test, y_te_col, period=F1_PERIOD, threshold=THRESHOLD)

    model = lgb.train(
        params=LGBM_PARAMS,
        train_set=dtrain,
        num_boost_round=N_ESTIMATORS,
        valid_sets=[deval],
        callbacks=[
            lgb.early_stopping(EARLY_STOP, verbose=False),
            lgb.early_stopping(EARLY_STOP, verbose=False),
            f1_cb,
        ],
    )

    models.append(model)

    # Finalne F1 dla tej klasy na zbiorze testowym
    y_prob = model.predict(X_test).astype(np.float32)
    prob_matrix_test[:, i] = y_prob
    f1 = f1_score(y_te_col, (y_prob >= THRESHOLD).astype(int),
                  average="binary", zero_division=0)
    f1_per_class[class_name] = f1

    running_macro = float(np.mean(list(f1_per_class.values())))
    pbar.set_postfix({
        "klasa":    class_name,
        "F1_klasy": f"{f1:.4f}",
        "F1_macro": f"{running_macro:.4f}",
        "drzewa":   model.num_trees(),
    })

pbar.close()

elapsed        = time.time() - t0
final_macro_f1 = float(np.mean(list(f1_per_class.values())))

print(f"\n{'='*55}")
print(f"  TRENING ZAKOŃCZONY  ({elapsed/60:.1f} min)")
print(f"  Macro-averaged F1 (test): {final_macro_f1:.4f}")
print(f"{'='*55}")

sorted_f1 = sorted(f1_per_class.items(), key=lambda x: x[1])
print("\nNajgorsze 5 klas:")
for name, score in sorted_f1[:5]:
    print(f"  {name}: {score:.4f}")
print("\nNajlepsze 5 klas:")
for name, score in sorted_f1[-5:]:
    print(f"  {name}: {score:.4f}")


Sukces: ipywidgets zaimportowane bez restartu!







Trening klas:   0%|          | 0/500 [00:00<?, ?klasa/s]




Trening klas:   0%|          | 0/500 [00:02<?, ?klasa/s, klasa=class_0, F1_klasy=1.0000, F1_macro=1.0000, drzewa=1]




Trening klas:   0%|          | 1/500 [00:02<24:56,  3.00s/klasa, klasa=class_0, F1_klasy=1.0000, F1_macro=1.0000, drzewa=1]

[1]	valid_0's binary_logloss: 0
[2]	valid_0's binary_logloss: 0
[3]	valid_0's binary_logloss: 0
[4]	valid_0's binary_logloss: 0
[5]	valid_0's binary_logloss: 0
[6]	valid_0's binary_logloss: 0
[7]	valid_0's binary_logloss: 0
[8]	valid_0's binary_logloss: 0
[9]	valid_0's binary_logloss: 0
[10]	valid_0's binary_logloss: 0
[11]	valid_0's binary_logloss: 0
[12]	valid_0's binary_logloss: 0
[13]	valid_0's binary_logloss: 0
[14]	valid_0's binary_logloss: 0
[15]	valid_0's binary_logloss: 0
[16]	valid_0's binary_logloss: 0
[17]	valid_0's binary_logloss: 0
[18]	valid_0's binary_logloss: 0
[19]	valid_0's binary_logloss: 0
[20]	valid_0's binary_logloss: 0
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[21]	valid_0's binary_logloss: 0
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[22]	valid_0's binary_logloss: 0
[LightGBM] [Warning] Stopped training because there are no more leaves that






Trening klas:   0%|          | 1/500 [00:08<24:56,  3.00s/klasa, klasa=class_1, F1_klasy=0.9988, F1_macro=0.9994, drzewa=77]




Trening klas:   0%|          | 2/500 [00:08<36:37,  4.41s/klasa, klasa=class_1, F1_klasy=0.9988, F1_macro=0.9994, drzewa=77]


[106]	valid_0's binary_logloss: 0.0160907
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[107]	valid_0's binary_logloss: 0.0161291
[1]	valid_0's binary_logloss: 0.0866751
[2]	valid_0's binary_logloss: 0.0848415
[3]	valid_0's binary_logloss: 0.0840685
[4]	valid_0's binary_logloss: 0.0836208
[5]	valid_0's binary_logloss: 0.0828395
[6]	valid_0's binary_logloss: 0.0832251
[7]	valid_0's binary_logloss: 0.0831984
[8]	valid_0's binary_logloss: 0.0836776
[9]	valid_0's binary_logloss: 0.0839725
[10]	valid_0's binary_logloss: 0.0836089
[11]	valid_0's binary_logloss: 0.0827259
[12]	valid_0's binary_logloss: 0.0824719
[13]	valid_0's binary_logloss: 0.081908
[14]	valid_0's binary_logloss: 0.0814966
[15]	valid_0's binary_logloss: 0.0808376
[16]	valid_0's binary_logloss: 0.0809013
[17]	valid_0's binary_logloss: 0.0803889
[18]	valid_0's binary_logloss: 0.0802883
[19]	valid_0's binary_logloss: 0.0804135
[20]	valid_0's binary_logloss: 0.0799439
[21]	valid_0's binary_logloss:

...


[253]	valid_0's binary_logloss: 0.0248841
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[254]	valid_0's binary_logloss: 0.0248854
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[255]	valid_0's binary_logloss: 0.0248754
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[256]	valid_0's binary_logloss: 0.0248823
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[257]	valid_0's binary_logloss: 0.0248721
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[258]	valid_0's binary_logloss: 0.0248673
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[259]	valid_0's binary_logloss: 0.024876
[1]	valid_0's binary_logloss: 1.54695
[2]	valid_0's binary_logloss: 0.275842
[3]	valid_0's binary_logloss: 1.07543
[4]	valid_0's binary_logloss: 1.01748
[5]	valid_0's binary_logloss: 1.01637
[6]	valid_0's binary_logloss: 1.43875
[7]	valid_0's binary_loglos






Trening klas: 100%|█████████▉| 498/500 [56:29<00:13,  6.59s/klasa, klasa=class_498, F1_klasy=0.0000, F1_macro=0.6032, drzewa=2]  




Trening klas: 100%|█████████▉| 499/500 [56:29<00:05,  5.75s/klasa, klasa=class_498, F1_klasy=0.0000, F1_macro=0.6032, drzewa=2]

[32]	valid_0's binary_logloss: 1.40949
[1]	valid_0's binary_logloss: 0.370678
[2]	valid_0's binary_logloss: 0.248573
[3]	valid_0's binary_logloss: 0.236462
[4]	valid_0's binary_logloss: 0.226897
[5]	valid_0's binary_logloss: 0.221886
[6]	valid_0's binary_logloss: 0.220913
[7]	valid_0's binary_logloss: 0.213429
[8]	valid_0's binary_logloss: 0.207237
[9]	valid_0's binary_logloss: 0.202789
[10]	valid_0's binary_logloss: 0.197378
[11]	valid_0's binary_logloss: 0.192185
[12]	valid_0's binary_logloss: 0.188466
[13]	valid_0's binary_logloss: 0.184514
[14]	valid_0's binary_logloss: 0.180584
[15]	valid_0's binary_logloss: 0.177474
[16]	valid_0's binary_logloss: 0.174315
[17]	valid_0's binary_logloss: 0.171185
[18]	valid_0's binary_logloss: 0.16794
[19]	valid_0's binary_logloss: 0.165453
[20]	valid_0's binary_logloss: 0.162576
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[21]	valid_0's binary_logloss: 0.159789
[LightGBM] [Warning] No further splits with positive gai






Trening klas: 100%|█████████▉| 499/500 [56:38<00:05,  5.75s/klasa, klasa=class_499, F1_klasy=0.4324, F1_macro=0.6028, drzewa=300]




Trening klas: 100%|██████████| 500/500 [56:38<00:00,  6.80s/klasa, klasa=class_499, F1_klasy=0.4324, F1_macro=0.6028, drzewa=300]



  TRENING ZAKOŃCZONY  (56.6 min)
  Macro-averaged F1 (test): 0.6028

Najgorsze 5 klas:
  class_106: 0.0000
  class_121: 0.0000
  class_123: 0.0000
  class_162: 0.0000
  class_167: 0.0000

Najlepsze 5 klas:
  class_244: 1.0000
  class_246: 1.0000
  class_247: 1.0000
  class_290: 1.0000
  class_421: 1.0000


In [122]:
# ============================================================
# CELL 8 — Ewaluacja końcowa + propagacja DAG
# ============================================================
y_pred_raw = (prob_matrix_test >= THRESHOLD).astype(np.int8)
y_pred_dag = propagate_labels_upward(y_pred_raw, dag, label_cols)

f1_before = f1_score(y_test, y_pred_raw, average="macro", zero_division=0)
f1_after  = f1_score(y_test, y_pred_dag, average="macro", zero_division=0)
incons    = count_dag_inconsistencies(prob_matrix_test, dag, label_cols)

print(f"\nF1 macro — bez propagacji DAG : {f1_before:.4f}")
print(f"F1 macro — po  propagacji DAG : {f1_after:.4f}  (+{f1_after - f1_before:.4f})")
print(f"Niespójności DAG : {incons}")


F1 macro — bez propagacji DAG : 0.6028
F1 macro — po  propagacji DAG : 0.5832  (+-0.0196)
Niespójności DAG              : 1137628


In [123]:
# ============================================================
# CELL 9 — Analiza wyników per klasa
# ============================================================
import pandas as pd
from sklearn.metrics import precision_score, recall_score

rows = []
for i, class_name in enumerate(label_cols):
    y_true_col = y_test[:, i]
    y_pred_col = y_pred_dag[:, i]
    y_prob_col = prob_matrix_test[:, i]

    n_positive = int(y_true_col.sum())
    n_predicted = int(y_pred_col.sum())

    f1  = f1_score(y_true_col, y_pred_col, average="binary", zero_division=0)
    pre = precision_score(y_true_col, y_pred_col, zero_division=0)
    rec = recall_score(y_true_col, y_pred_col, zero_division=0)

    rows.append({
        "klasa":        class_name,
        "f1":           round(f1, 4),
        "precision":    round(pre, 4),
        "recall":       round(rec, 4),
        "n_pozytywnych": n_positive,
        "n_przewidzianych": n_predicted,
        "prev_%":       round(100 * n_positive / len(y_true_col), 2),
    })

df_results = pd.DataFrame(rows).sort_values("f1", ascending=True).reset_index(drop=True)

In [126]:
df_results

,klasa,f1,precision,recall,n_pozytywnych,n_przewidzianych,prev_%
0,class_386,0.0,0.0,0.0,0,2,0.00
1,class_416,0.0,0.0,0.0,0,0,0.00
2,class_420,0.0,0.0,0.0,0,0,0.00
3,class_384,0.0,0.0,0.0,1,7,0.01
4,class_300,0.0,0.0,0.0,0,0,0.00
...,...,...,...,...,...,...,...
495,class_246,1.0,1.0,1.0,1,1,0.01
496,class_247,1.0,1.0,1.0,1,1,0.01
497,class_290,1.0,1.0,1.0,1,1,0.01
498,class_191,1.0,1.0,1.0,52,52,0.77


In [125]:
df_results.to_csv("results.csv", index = False)

In [121]:
import joblib
import os

# Tworzymy folder na model, jeśli nie istnieje
os.makedirs("model_output", exist_ok=True)

# Budujemy słownik z kompletnym stanem modelu
model_artifact = {
    "models": models,            # Lista 500 modeli Booster
    "label_cols": label_cols,    # Nazwy klas (kluczowe do predict!)
    "threshold": THRESHOLD       # Próg decyzyjny użyty w treningu
}

# Zapisujemy wszystko do jednego skompresowanego pliku
model_filename = "model_output/chebi_lgbm_ensemble.joblib"
joblib.dump(model_artifact, model_filename, compress=3)

print(f"Sukces! Model zapisany w: {model_filename}")

Exception ignored in: <function tqdm.__del__ at 0x00000200679A6950>
Traceback (most recent call last):
  File "C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\tqdm\std.py", line 1145, in __del__
    self.close()
  File "C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\tqdm\notebook.py", line 283, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Exception ignored in: <function tqdm.__del__ at 0x00000200679A6950>
Traceback (most recent call last):
  File "C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\tqdm\std.py", line 1145, in __del__
    self.close()
  File "C:\Users\domin\PycharmProjects\ml-train-lgbm\.venv\lib\site-packages\tqdm\notebook.py", line 283, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Exception ignored in: <function tqdm.__del__ at 0x00000200679A6950>
Trac

Sukces! Model zapisany w: model_output/chebi_lgbm_ensemble.joblib
